# Módulo I5 · Deep learning — Soluciones de los ejercicios

Este notebook resuelve los cuatro ejercicios del final de `notebook.ipynb`. Es **autocontenido**: reproduce la preparación de datos y
las funciones auxiliares del notebook original (con un pequeño cambio: `entrenar` acepta ahora el *DataLoader* de entrenamiento
como argumento, lo que necesitamos para el ejercicio de aumento de datos).

Todo funciona en CPU en pocos minutos.

> Recuerda: con 360 imágenes de test, **un solo error equivale a 0,28 puntos de exactitud**. Diferencias de una o dos décimas
> entre modelos pueden deberse al azar de la inicialización. Por eso, cuando comparamos variantes, repetimos con varias semillas.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

sns.set_theme(style="whitegrid")
torch.manual_seed(0)
torch.set_num_threads(1)   # con redes tan pequeñas, un hilo es lo más rápido (evita la sobrecarga de paralelizar)
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
print("Usando:", dispositivo)

## Preparación (igual que en el notebook)

In [ ]:
digits = load_digits()
X = (digits.images / 16.0).astype(np.float32)   # píxeles en [0, 1]
y = digits.target
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=0.2, stratify=y_tmp, random_state=0)
print(f"train: {len(X_train)} | validación: {len(X_val)} | test: {len(X_test)}")

def loader(X, y, batch=64, shuffle=False):
    return DataLoader(TensorDataset(torch.tensor(X).unsqueeze(1), torch.tensor(y)), batch_size=batch, shuffle=shuffle)

dl_train, dl_val, dl_test = loader(X_train, y_train, shuffle=True), loader(X_val, y_val), loader(X_test, y_test)

base = LogisticRegression(max_iter=2000).fit(X_train.reshape(len(X_train), -1), y_train)
acc_base = base.score(X_test.reshape(len(X_test), -1), y_test)
print(f"Regresión logística (sklearn) – exactitud test: {acc_base:.3f}")

In [ ]:
def exactitud(modelo, dl):
    modelo.eval()
    aciertos = 0
    with torch.no_grad():
        for xb, yb in dl:
            aciertos += (modelo(xb.to(dispositivo)).argmax(1).cpu() == yb).sum().item()
    return aciertos / len(dl.dataset)

def entrenar(modelo, epocas=40, lr=1e-3, paciencia=None, dl_tr=None):
    """Igual que en el notebook, pero con `dl_tr` opcional (por defecto, dl_train).
    `dl_tr` puede ser un DataLoader o una función sin argumentos que devuelve un DataLoader nuevo en cada época."""
    dl_tr = dl_train if dl_tr is None else dl_tr
    modelo.to(dispositivo)
    opt = torch.optim.Adam(modelo.parameters(), lr=lr)
    perdida_fn = nn.CrossEntropyLoss()
    hist = {"perdida_train": [], "acc_train": [], "acc_val": []}
    mejor, mejor_estado, sin_mejora = 0, None, 0
    for _ in range(epocas):
        dl_epoca = dl_tr() if callable(dl_tr) else dl_tr
        modelo.train()
        total = 0.0
        for xb, yb in dl_epoca:
            xb, yb = xb.to(dispositivo), yb.to(dispositivo)
            opt.zero_grad()
            perdida = perdida_fn(modelo(xb), yb)
            perdida.backward()      # retropropagación
            opt.step()              # actualización de pesos
            total += perdida.item() * len(xb)
        hist["perdida_train"].append(total / len(dl_epoca.dataset))
        hist["acc_train"].append(exactitud(modelo, dl_train))   # siempre sobre las imágenes originales
        hist["acc_val"].append(exactitud(modelo, dl_val))
        if paciencia is not None:   # early stopping
            if hist["acc_val"][-1] > mejor:
                mejor, sin_mejora = hist["acc_val"][-1], 0
                mejor_estado = {k: v.clone() for k, v in modelo.state_dict().items()}
            else:
                sin_mejora += 1
                if sin_mejora >= paciencia:
                    break
    if mejor_estado is not None:
        modelo.load_state_dict(mejor_estado)
    return hist

def curvas(hist, titulo):
    fig, axes = plt.subplots(1, 2, figsize=(11, 3))
    axes[0].plot(hist["perdida_train"]); axes[0].set(title="Pérdida (train)", xlabel="época")
    axes[1].plot(hist["acc_train"], label="train"); axes[1].plot(hist["acc_val"], label="validación")
    axes[1].set(title="Exactitud", xlabel="época"); axes[1].legend()
    fig.suptitle(titulo); plt.tight_layout(); plt.show()

def n_param(modelo):
    return sum(p.numel() for p in modelo.parameters())

---
## Ejercicio 1 · MLP sin capas ocultas

*Entrena el MLP sin capas ocultas (`nn.Flatten(), nn.Linear(64, 10)`). ¿A qué modelo clásico equivale?*

In [ ]:
torch.manual_seed(0)
lineal = nn.Sequential(nn.Flatten(), nn.Linear(64, 10))
print("Parámetros:", n_param(lineal), "= 64 × 10 pesos + 10 sesgos")
t0 = time.time()
hist_lineal = entrenar(lineal, epocas=150)
print(f"Tiempo de entrenamiento: {time.time() - t0:.1f} s")
curvas(hist_lineal, "Red sin capas ocultas (= regresión logística multinomial)")
acc_lineal = exactitud(lineal, dl_test)
print(f"Red sin capas ocultas – exactitud test: {acc_lineal:.3f}")
print(f"Regresión logística (sklearn) – exactitud test: {acc_base:.3f}")

Comprobemos también hasta qué punto **coinciden las predicciones** de ambos modelos, no solo su exactitud:

In [ ]:
X_test_t = torch.tensor(X_test).unsqueeze(1)
lineal.eval()
with torch.no_grad():
    pred_lineal = lineal(X_test_t).argmax(1).numpy()
pred_base = base.predict(X_test.reshape(len(X_test), -1))
print(f"Predicciones idénticas en el {np.mean(pred_lineal == pred_base):.1%} de las imágenes de test")

**Interpretación.** Una red con una sola capa lineal seguida de la entropía cruzada (`nn.CrossEntropyLoss` aplica internamente
*softmax* + log-verosimilitud negativa) es exactamente una **regresión logística multinomial** (*softmax regression*): calcula
$z = Wx + b$ con 10 salidas y convierte esas puntuaciones en probabilidades con *softmax*. El modelo tiene 650 parámetros
(64 × 10 + 10) y su frontera de decisión entre cada par de clases es lineal.

La exactitud obtenida (≈ 0,95) queda muy cerca de la de `LogisticRegression` de scikit-learn (0,964): apenas unos pocos
aciertos de diferencia, y las predicciones coinciden en torno al 98 % de las imágenes. Las pequeñas diferencias vienen de
*cómo* se ajusta, no de *qué* modelo es: scikit-learn usa por defecto
regularización L2 ($C=1$) y un optimizador de segundo orden (L-BFGS) hasta converger, mientras que aquí usamos Adam con
minilotes, sin regularización y un número fijo de épocas.

Moraleja: lo que hace "profunda" a una red no es PyTorch, sino **las capas ocultas con activaciones no lineales**.

---
## Ejercicio 2 · MLP el doble de ancho, 150 épocas, sin early stopping

*Duplica el número de neuronas del MLP y entrena 150 épocas sin early stopping. ¿Ves sobreajuste en las curvas?*

In [ ]:
torch.manual_seed(0)
mlp_grande = nn.Sequential(
    nn.Flatten(),
    nn.Linear(64, 256), nn.ReLU(),
    nn.Linear(256, 128), nn.ReLU(),
    nn.Linear(128, 10),
)
print("Parámetros MLP grande:", n_param(mlp_grande), "(el MLP original tenía 17 226)")
t0 = time.time()
hist_grande = entrenar(mlp_grande, epocas=150)
print(f"Tiempo de entrenamiento: {time.time() - t0:.1f} s")
curvas(hist_grande, "MLP ×2 neuronas, 150 épocas, sin early stopping")

In [ ]:
# Pérdida de VALIDACIÓN por época: la vemos reentrenando el mismo modelo y midiéndola en cada época
def perdida(modelo, dl):
    modelo.eval()
    fn, total = nn.CrossEntropyLoss(reduction="sum"), 0.0
    with torch.no_grad():
        for xb, yb in dl:
            total += fn(modelo(xb), yb).item()
    return total / len(dl.dataset)

torch.manual_seed(0)
mlp_grande2 = nn.Sequential(nn.Flatten(), nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 10))
opt = torch.optim.Adam(mlp_grande2.parameters(), lr=1e-3)
fn = nn.CrossEntropyLoss()
perd_tr, perd_val = [], []
for _ in range(150):
    mlp_grande2.train()
    for xb, yb in dl_train:
        opt.zero_grad(); fn(mlp_grande2(xb), yb).backward(); opt.step()
    perd_tr.append(perdida(mlp_grande2, dl_train)); perd_val.append(perdida(mlp_grande2, dl_val))

ep_min = int(np.argmin(perd_val))
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(perd_tr, label="train"); ax.plot(perd_val, label="validación")
ax.axvline(ep_min, ls="--", c="grey"); ax.set_yscale("log")
ax.set(title="Pérdida (escala log) del MLP grande", xlabel="época"); ax.legend(); plt.tight_layout(); plt.show()

print(f"Exactitud train final: {hist_grande['acc_train'][-1]:.3f}")
print(f"Exactitud validación: máx. {max(hist_grande['acc_val']):.3f} (época {int(np.argmax(hist_grande['acc_val'])) + 1}) | final {hist_grande['acc_val'][-1]:.3f}")
print(f"Pérdida validación: mínima {perd_val[ep_min]:.3f} en la época {ep_min + 1} | final {perd_val[-1]:.3f}")
print(f"Pérdida train final: {perd_tr[-1]:.5f}")
print(f"MLP grande – exactitud test: {exactitud(mlp_grande, dl_test):.3f}")

**Interpretación.** Las curvas muestran la "firma" del sobreajuste, aunque de forma suave:

- La **exactitud de entrenamiento llega al 100 %** en pocas épocas y la **pérdida de entrenamiento** sigue cayendo hasta valores
  casi nulos: con unos 50 000 parámetros y 1 149 imágenes, la red puede memorizar el conjunto de entrenamiento.
- La **pérdida de validación** alcanza su mínimo pronto (ver la línea discontinua) y después **sube** de forma sostenida: el modelo
  se vuelve cada vez más confiado, también cuando se equivoca. Esta es la señal más clara de sobreajuste.
- La **exactitud de validación**, en cambio, se estanca y oscila en torno a su máximo en lugar de desplomarse. Con datos tan
  limpios, sobreajustar no destruye la exactitud, pero sí la **calibración** de las probabilidades (algo importante si el
  negocio usa la confianza del modelo para decidir qué paquetes revisar a mano).

¿Qué harías? *Early stopping* sobre la pérdida o la exactitud de validación (quedarte con el modelo de la época del mínimo),
*weight decay* o *dropout*, o simplemente volver a una red más pequeña: el MLP original rinde igual con un tercio de parámetros.

---
## Ejercicio 3 · Tasa de aprendizaje: 1e-4, 1e-3 y 1e-2

*¿Qué ocurre con la curva de pérdida en cada caso?*

In [ ]:
def nuevo_mlp():
    return nn.Sequential(nn.Flatten(), nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 10))

resultados_lr = {}
for lr in [1e-4, 1e-3, 1e-2, 1e-1]:   # 1e-1 es un extra para ver qué pasa con una tasa excesiva
    torch.manual_seed(0)
    m = nuevo_mlp()
    h = entrenar(m, epocas=40, lr=lr)
    resultados_lr[lr] = (h, exactitud(m, dl_test))

fig, axes = plt.subplots(1, 2, figsize=(12, 3.2))
for lr, (h, _) in resultados_lr.items():
    axes[0].plot(h["perdida_train"], label=f"lr = {lr:g}")
    axes[1].plot(h["acc_val"], label=f"lr = {lr:g}")
axes[0].set(title="Pérdida de entrenamiento (escala log)", xlabel="época", yscale="log"); axes[0].legend()
axes[1].set(title="Exactitud de validación", xlabel="época", ylim=(0, 1.0)); axes[1].legend()
plt.tight_layout(); plt.show()

print(f"{'lr':>8} {'pérdida ép. 1':>14} {'pérdida ép. 10':>15} {'pérdida final':>14} {'acc val final':>14} {'acc test':>9}")
for lr, (h, acc_t) in resultados_lr.items():
    print(f"{lr:>8g} {h['perdida_train'][0]:>14.3f} {h['perdida_train'][9]:>15.3f} {h['perdida_train'][-1]:>14.4f} {h['acc_val'][-1]:>14.3f} {acc_t:>9.3f}")

**Interpretación** (compara con la tabla de arriba):

- **lr = 1e-4 (demasiado pequeña):** la pérdida baja de forma lenta y suave; a las 40 épocas el modelo todavía no ha convergido
  (pérdida final claramente más alta y exactitud algo menor). Necesitaría muchas más épocas: es **caro**, no **malo**.
- **lr = 1e-3 (valor por defecto de Adam):** caída rápida y estable; es el compromiso razonable entre velocidad y estabilidad.
- **lr = 1e-2 (grande):** la pérdida cae muy deprisa al principio, pero la curva es más **ruidosa**, con repuntes: los pasos son tan
  grandes que el optimizador "salta" alrededor del mínimo (fíjate en las subidas de la pérdida hacia las épocas 10–17). En este
  problema pequeño acaba convergiendo, e incluso llega antes a una pérdida casi nula.
- **lr = 1e-1 (extra, excesiva):** la pérdida se queda **estancada** muy por encima de las demás, con altibajos, y la exactitud
  se resiente claramente: los pasos son tan grandes que el optimizador no consigue bajar al fondo del valle.

Regla práctica: empieza con 1e-3 para Adam, mira la curva de pérdida y ajusta en factores de 10 (o de 3). Si la curva es muy
plana, sube la tasa; si es errática o explota, bájala.

---
## Ejercicio 4 · Aumento de datos con desplazamientos de ±1 píxel

*Desplaza aleatoriamente cada imagen de entrenamiento ±1 píxel (`np.roll`) y reentrena la CNN. ¿Mejora la exactitud de test?*

Generamos un **desplazamiento aleatorio nuevo en cada época** (en horizontal y vertical, cada uno en {-1, 0, +1}). Así la red ve
versiones distintas de cada dígito durante el entrenamiento. Validación y test **no** se aumentan: queremos medir el
rendimiento sobre imágenes reales.

`np.roll` es circular (lo que sale por un borde entra por el opuesto). En estas imágenes de 8×8 los bordes son casi siempre
blancos, así que el efecto es despreciable.

In [ ]:
rng = np.random.default_rng(0)

def desplazar(imgs, rng):
    salida = np.empty_like(imgs)
    dy = rng.integers(-1, 2, size=len(imgs))
    dx = rng.integers(-1, 2, size=len(imgs))
    for i, (img, a, b) in enumerate(zip(imgs, dy, dx)):
        salida[i] = np.roll(img, shift=(a, b), axis=(0, 1))
    return salida

def dl_train_aumentado():
    return loader(desplazar(X_train, rng), y_train, shuffle=True)

# Ejemplos visuales
ej = desplazar(X_train[:6], np.random.default_rng(1))
fig, axes = plt.subplots(2, 6, figsize=(9, 3))
for j in range(6):
    axes[0, j].imshow(X_train[j], cmap="gray_r"); axes[0, j].axis("off")
    axes[1, j].imshow(ej[j], cmap="gray_r"); axes[1, j].axis("off")
axes[0, 0].set_title("original", fontsize=9, loc="left"); axes[1, 0].set_title("desplazada", fontsize=9, loc="left")
plt.tight_layout(); plt.show()

In [ ]:
def nueva_cnn():
    return nn.Sequential(
        nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Flatten(),
        nn.Dropout(0.3),
        nn.Linear(32 * 2 * 2, 10),
    )

# Imágenes de test desplazadas: ¿es el modelo robusto a un paquete mal centrado?
X_test_despl = desplazar(X_test, np.random.default_rng(123))
dl_test_despl = loader(X_test_despl, y_test)

semillas = [0, 1, 2]
res = {"sin aumento": [], "con aumento": []}
hist_ej = {}
t0 = time.time()
for s in semillas:
    for nombre, dl_tr in [("sin aumento", dl_train), ("con aumento", dl_train_aumentado)]:
        torch.manual_seed(s)
        m = nueva_cnn()
        h = entrenar(m, epocas=80, paciencia=10, dl_tr=dl_tr)
        res[nombre].append((exactitud(m, dl_test), exactitud(m, dl_test_despl), len(h["acc_val"])))
        if s == 0:
            hist_ej[nombre] = h
print(f"Tiempo total (6 entrenamientos): {time.time() - t0:.0f} s\n")

print(f"{'':12s} {'semilla':>8} {'test':>7} {'test desplazado':>16} {'épocas':>7}")
for nombre, filas in res.items():
    for s, (a, b, e) in zip(semillas, filas):
        print(f"{nombre:12s} {s:>8} {a:>7.3f} {b:>16.3f} {e:>7}")
print()
for nombre, filas in res.items():
    a = np.array([f[0] for f in filas]); b = np.array([f[1] for f in filas])
    print(f"{nombre:12s} media test = {a.mean():.3f} ± {a.std():.3f} | media test desplazado = {b.mean():.3f} ± {b.std():.3f}")

In [ ]:
curvas(hist_ej["con aumento"], "CNN con aumento de datos (semilla 0)")

**Interpretación.**

- En el **test original**, el aumento de datos **no mejora** la exactitud de forma apreciable: las medias de las tres semillas son
  prácticamente iguales (≈ 0,97 en ambos casos) y la diferencia entre semillas es del mismo orden. Ojo: con una sola semilla
  (la 0) habríamos "visto" una mejora (0,967 → 0,975) que en realidad es ruido. No vendas una mejora basada en una sola ejecución.
  La razón es que las imágenes de test de este conjunto están tan bien centradas como las de entrenamiento.
- Donde el efecto es **claro** es en el **test desplazado** (imágenes de test movidas ±1 píxel, como un código postal mal
  centrado): la CNN entrenada sin aumento se desploma hasta ≈ 0,55, mientras que la entrenada con aumento se mantiene en
  ≈ 0,91. El aumento de datos enseña a la red una **invariancia** que nos interesa en producción. (Que la CNN original sea
  tan frágil sorprende: en imágenes de 8×8, un píxel es el 12 % del ancho, y tras dos *max pooling* la capa final aún
  depende de la posición.)
- En las curvas, la exactitud de entrenamiento con aumento sube más despacio y la distancia entre entrenamiento y validación se
  reduce: el aumento actúa como **regularizador**. A cambio, el entrenamiento suele necesitar más épocas.

**Para el informe de logística:** el aumento de datos es barato (no requiere etiquetar nada nuevo) y hace el modelo más robusto
a variaciones realistas (posición, y por extensión rotación o grosor del trazo). Antes de afirmar que "mejora la exactitud",
evalúa con varias semillas y, mejor aún, con datos que se parezcan a los de producción.

---
## Resumen de los ejercicios

| Ejercicio | Conclusión |
|---|---|
| 1. Red sin capas ocultas | Es una regresión logística multinomial; exactitud muy próxima a la de scikit-learn (≈ 98 % de predicciones iguales). |
| 2. MLP ×2, 150 épocas | Train al 100 % y pérdida de validación que sube tras su mínimo: sobreajuste (suave en exactitud, claro en pérdida). |
| 3. Tasa de aprendizaje | 1e-4 lenta, 1e-3 rápida y estable, 1e-2 rápida pero con repuntes, 1e-1 se estanca. |
| 4. Aumento de datos | Sin mejora en el test original (≈ 0,97); en imágenes desplazadas, de ≈ 0,55 a ≈ 0,91. |